In [ ]:
# Import libraries
from delta import DeltaTable
from pyspark.sql.functions import min, max, col, sequence, explode, dayofweek, date_format, weekofyear, month, quarter, year, concat, lit
import json

In [ ]:
# Define parameters with None, as values will come from pipeline activities
# Source
source_lakehouse_path = None 

# Target
target_lakehouse_path = None 
target_schema = None
target_table = None

In [ ]:
# Define variables using values from parameters
source_path = f'{source_lakehouse_path}/Tables'
target_path = f'{target_lakehouse_path}/Tables/{target_schema}/{target_table}'

# Load Helper Notebooks

Load helper notebook with the schema, merge keys and indicator mapping definition of gold tables inside "gold_objects" dictionary: 


In [ ]:
%run ./NB_Gold_Table_Schemas

In [ ]:
# Use gold_objects dictionary to define aditional variables
target_table_schema = gold_objects[f"{target_table}"]["schema"]

# Read and Transform New Data

In [ ]:
# Read source tables from Silver
generation_df = spark.read.format('delta').load(f'{source_path}/dbo/generation')
generation_forecast_df = spark.read.format('delta').load(f'{source_path}/dbo/generation_forecast')

In [ ]:
# Min/Max dates
start_date = (generation_df.select(min("datetime_locale").alias("min_date")).first()["min_date"])
end_date = (generation_forecast_df.select(max("datetime_locale").alias("max_date")).first()["max_date"])

# Convert to date (start_date should start on january 1st)
start_date = start_date.replace(month=1, day=1).date() 
end_date = end_date.date() 

In [ ]:
# Create date range
df = spark.sql(f"""SELECT explode(sequence(to_date('{start_date}'), to_date('{end_date}'), interval 1 day)) AS date""")

In [ ]:
# Create calendar dimension
df = (
    df
    .withColumn("day_of_week", ((dayofweek(col("date")) + 5) % 7) + 1) # Monday = 1, Sunday = 7
    .withColumn("day_name", date_format(col("date"), "EEEE"))
    .withColumn("is_weekend", dayofweek(col("date")).isin([1, 7]))
    .withColumn("week_of_year", weekofyear(col("date")))
    .withColumn("month_number", month(col("date")))
    .withColumn("month_name", date_format(col("date"), "MMMM"))
    .withColumn("month_short_name", date_format(col("date"), "MMM"))
    .withColumn("quarter_number", quarter(col("date")))
    .withColumn("quarter_name", concat(lit("Q"), quarter(col("date"))))
    .withColumn("year", year(col("date")))
    .withColumn("year_month", date_format(col("date"), "yyyy-MM"))
    .withColumn("year_month_number",(year(col("date")) * 100) + month(col("date")))
    .select(
        "date",
        "day_of_week",
        "day_name",
        "is_weekend",
        "week_of_year",
        "month_number",
        "month_name",
        "month_short_name",
        "quarter_number",
        "quarter_name",
        "year",
        "year_month",
        "year_month_number"
    )
)

# Update Table

In [ ]:
# Full load
# Create the table if it doesnt exists
try:
    delta_table = DeltaTable.forPath(spark, target_path)
    print('Table already exists')
except:
    delta_table = DeltaTable.createIfNotExists(spark) \
    .location(target_path) \
    .addColumns(target_table_schema) \
    .execute()
    print('Table has been created')

# Recreate the table without infering schema
df.write.format("delta").mode("overwrite").save(target_path) 

# Statistics
rows_inserted = df.count()
rows_updated = 0

In [ ]:
# File a dictionary with operational statistics
result = {
    "maxdate": None,
    "rows_inserted": rows_inserted,
    "rows_updated": rows_updated
}

# Make the result values queriable for other activities in the pipeline
mssparkutils.notebook.exit(json.dumps(result))